In [15]:
from typing import Literal
import torch

device: Literal['cpu', 'cuda'] = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(torch.cuda.get_device_name())

NVIDIA GeForce MX350


In [16]:
from torch import nn
from torch.nn import Module

class Model(Module):
    def __init__(self, num_features):
        super().__init__() # Calling base class constructor is maindatory, everything happening under the hood gets initialized and managed by base class (parameter initialization and its state, BatchNorm state, cpu → gpu transfe, etc)

        """ Define all the individual layers of the model in the constructor """

        self.network = nn.Sequential(
            # Two types of models → sequential, functional (Residual connections)
            nn.Linear(in_features = num_features, out_features = 3),
            nn.BatchNorm1d(num_features=3),
            nn.ReLU(),
            nn.Linear(in_features = 3, out_features = 1), 
            # nn.Sigmoid() → No need to use any activation function on output layer → loss function automatically applies it before actually calculating the loss
        )
        # The model parameters are randomly initialized (default - glorat uniform) in float32 datatype → your input features must be in that same datatype, not even float64 or float16 allowed (implicite conversion does not happen)

    def forward(self, features):
        """ Connect the layers by passing the features and outputs to the next layers """
        out = self.network(features) # magic method __call__ → just pass the input features to object `obj(input)`
        return out

In [17]:
from torch._tensor import Tensor

# If you want to train your model on GPU, your model and data must be shifted on GPU memory.
features: Tensor = torch.rand(10, 5, requires_grad=False).to(device=device)
model: Model = Model(features.shape[1]).to(device=device)

"""
    torch.compile(model) is a just-in-time (JIT) compiler for PyTorch code. It does not change what your model computes; it changes how that computation is executed. The goal is to make the same mathematical operations run faster by generating optimized machine code, reducing Python overhead, and fusing operations together.

    Trade-offs:
        1. Best for repeated execution with stable shapes. The speedup is most pronounced when the same model runs many times with similar tensor shapes, which is typical for training loops and LLM inference.

        2. Not worth it for short-lived or highly dynamic workloads. If the model runs only a few times or shapes change on every call, the compilation cost may outweigh the benefit.

        3. The default CUDA compiler backend, Triton, requires CUDA capability 7.0 or newer.
"""
supports_compile: bool = (
    device != "cuda"
    or torch.cuda.get_device_capability(device)[0] >= 7
)
if supports_compile:
    model = torch.compile(model=model)
else:
    print(f"Skipping torch.compile: this CUDA GPU capability is {torch.cuda.get_device_capability()} which is below 7.0.")

model(features) # forward method through `__call__` magic method

Skipping torch.compile: this CUDA GPU capability is (6, 1) which is below 7.0.


tensor([[-0.1812],
        [ 0.0847],
        [-0.4074],
        [-0.3629],
        [-0.4492],
        [-0.2309],
        [-0.2139],
        [-0.0280],
        [ 0.1548],
        [ 0.1205]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [18]:
# model architecture
from torchinfo import summary

"""
    torchinfo.summary() does not perform any training. It runs exactly `one forward pass` through your model inside a `torch.no_grad() block`, then reads information from hooks attached to each layer. No backward pass is executed, no gradients are computed, and no optimizer step is taken. The “Forward/backward pass size” line is an analytical estimate that assumes a backward pass would happen, even though summary() itself never triggers one.
"""
summary(model, input_data=features, device=device) # `input_data` and `input_shape` are mutually exclusive

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─BatchNorm1d: 2-2                  [10, 3]                   6
│    └─ReLU: 2-3                         [10, 3]                   --
│    └─Linear: 2-4                       [10, 1]                   4
Total params: 28
Trainable params: 28
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

When you define a custom model by subclassing nn.Module, you almost always implement `__init__` and `forward`. But the base class provides a rich set of other methods that are essential for training, evaluation, saving/loading, device management, and debugging.

In [19]:
# Returns all the trainable and non-trainable parameters of all the layers in the module - save the model paremeters
model.state_dict()

OrderedDict([('network.0.weight',
              tensor([[-0.0670, -0.3597,  0.4409,  0.0183, -0.0505],
                      [-0.3281, -0.3866,  0.3180,  0.2063,  0.3302],
                      [-0.1031, -0.1257, -0.2789, -0.2031, -0.0818]], device='cuda:0')),
             ('network.0.bias',
              tensor([ 4.0795e-01,  9.5962e-05, -3.3242e-01], device='cuda:0')),
             ('network.1.weight', tensor([1., 1., 1.], device='cuda:0')),
             ('network.1.bias', tensor([0., 0., 0.], device='cuda:0')),
             ('network.1.running_mean',
              tensor([ 0.0451,  0.0025, -0.0707], device='cuda:0')),
             ('network.1.running_var',
              tensor([0.9032, 0.9081, 0.9017], device='cuda:0')),
             ('network.1.num_batches_tracked', tensor(1, device='cuda:0')),
             ('network.3.weight',
              tensor([[-0.2340, -0.0274,  0.3156]], device='cuda:0')),
             ('network.3.bias', tensor([-0.1812], device='cuda:0'))])

In [20]:
for name, parameters in model.named_parameters():
    # Returns an iterator over a model parameters
    print(f"{name}: {parameters}")

network.0.weight: Parameter containing:
tensor([[-0.0670, -0.3597,  0.4409,  0.0183, -0.0505],
        [-0.3281, -0.3866,  0.3180,  0.2063,  0.3302],
        [-0.1031, -0.1257, -0.2789, -0.2031, -0.0818]], device='cuda:0',
       requires_grad=True)
network.0.bias: Parameter containing:
tensor([ 4.0795e-01,  9.5962e-05, -3.3242e-01], device='cuda:0',
       requires_grad=True)
network.1.weight: Parameter containing:
tensor([1., 1., 1.], device='cuda:0', requires_grad=True)
network.1.bias: Parameter containing:
tensor([0., 0., 0.], device='cuda:0', requires_grad=True)
network.3.weight: Parameter containing:
tensor([[-0.2340, -0.0274,  0.3156]], device='cuda:0', requires_grad=True)
network.3.bias: Parameter containing:
tensor([-0.1812], device='cuda:0', requires_grad=True)


Buffers are tensors that are part of the module’s state but are not updated (non-trainable) by the optimizer. Classic examples are the running mean and variance in BatchNorm.

In [21]:
for name, buf in model.named_buffers():
    print(name, buf.shape)

network.1.running_mean torch.Size([3])
network.1.running_var torch.Size([3])
network.1.num_batches_tracked torch.Size([])


In [22]:
for child in model.named_children():
    # Just one level inside. Use model.modules() for iterating over entire model
    print(child)

('network', Sequential(
  (0): Linear(in_features=5, out_features=3, bias=True)
  (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU()
  (3): Linear(in_features=3, out_features=1, bias=True)
))


In [23]:
# Weight initialization
for m in model.modules():
    print(m)
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)

Model(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=3, bias=True)
    (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=3, out_features=1, bias=True)
  )
)
Sequential(
  (0): Linear(in_features=5, out_features=3, bias=True)
  (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU()
  (3): Linear(in_features=3, out_features=1, bias=True)
)
Linear(in_features=5, out_features=3, bias=True)
BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
ReLU()
Linear(in_features=3, out_features=1, bias=True)


In [24]:
# Weight initialization using apply metho
@torch.no_grad()
def init_weights(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(tensor=module.weight)

model = model.apply(init_weights)
model

Model(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=3, bias=True)
    (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=3, out_features=1, bias=True)
  )
)

In [25]:
model.add_module(
    name="temprary_module",
    module=nn.Linear(in_features=1, out_features=1)
) # Append at the last as new block

for child in model.named_children():
    print(child)

for name, parameters in model.named_parameters():
    # Returns the trainable and non-trainable parameters from all main and nested layers
    print(f"{name}: {parameters}")

('network', Sequential(
  (0): Linear(in_features=5, out_features=3, bias=True)
  (1): BatchNorm1d(3, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU()
  (3): Linear(in_features=3, out_features=1, bias=True)
))
('temprary_module', Linear(in_features=1, out_features=1, bias=True))
network.0.weight: Parameter containing:
tensor([[-0.0473, -0.2680,  0.1065, -0.0821, -0.3593],
        [ 0.6677,  0.0344, -0.3015,  0.2590, -0.3400],
        [-0.4254, -1.0385,  0.3646, -0.1545, -0.6041]], device='cuda:0',
       requires_grad=True)
network.0.bias: Parameter containing:
tensor([ 4.0795e-01,  9.5962e-05, -3.3242e-01], device='cuda:0',
       requires_grad=True)
network.1.weight: Parameter containing:
tensor([1., 1., 1.], device='cuda:0', requires_grad=True)
network.1.bias: Parameter containing:
tensor([0., 0., 0.], device='cuda:0', requires_grad=True)
network.3.weight: Parameter containing:
tensor([[ 0.7755, -0.1574, -0.2372]], device='cuda:0', requires_gr

In [26]:
model.get_submodule(target="temprary_module") # Instance of a perticular layer

Linear(in_features=1, out_features=1, bias=True)

In [27]:
# Attributes of layer instance
print(model.get_submodule(target="temprary_module").weight)
print(model.get_submodule(target="temprary_module").bias)

Parameter containing:
tensor([[0.0936]], requires_grad=True)
Parameter containing:
tensor([-0.5968], requires_grad=True)
